# ComfyUI + Z-Image Turbo - 2560x1440 生成

## 手順
1. ランタイム → GPU T4以上を選択
2. すべてのセルを実行
3. HuggingFace tokenを求められたら入力

In [ ]:
# GPU確認
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# HuggingFace token入力
print("HuggingFace tokenを入力してください")
print("(https://huggingface.co/settings/tokens で取得可能)")
HF_TOKEN = input("token: ")

In [ ]:
# ComfyUIインストール
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -r requirements.txt -q

In [ ]:
# Z-Image Turboモデルをダウンロード
import os
from huggingface_hub import login, hf_hub_download

os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)

login(token=HF_TOKEN)

print("Downloading Z-Image Turbo model (FP8, ~10GB)...")

# SeeSee21/Z-Image-Turbo-AIO からダウンロード
hf_hub_download(
    repo_id="SeeSee21/Z-Image-Turbo-AIO",
    filename="z-image-turbo-fp8-aio.safetensors",
    local_dir="/content/ComfyUI/models/checkpoints"
)

# ファイル確認
size = os.path.getsize("/content/ComfyUI/models/checkpoints/z-image-turbo-fp8-aio.safetensors")
print(f"Download complete! File size: {size / (1024**3):.2f} GB")

In [ ]:
# ComfyUI起動
import threading
import subprocess
import time

def run_comfyui():
    subprocess.Popen(["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"])

t = threading.Thread(target=run_comfyui, daemon=True)
t.start()
print("ComfyUI starting...")
time.sleep(20)
print("Ready!")

In [ ]:
# 接続確認とモデル一覧取得
import requests

# ComfyUIが起動するまで待機
for i in range(10):
    try:
        r = requests.get("http://127.0.0.1:8188/system_stats", timeout=5)
        print("ComfyUI connected!")
        break
    except:
        print(f"Waiting... {i+1}/10")
        time.sleep(3)

# 利用可能なモデル一覧取得
models = requests.get("http://127.0.0.1:8188/api/models").json()
print("\n利用可能なcheckpoint:")
for m in models:
    if "checkpoints" in m.get("model_type", ""):
        print(f"  - {m.get('name', 'unknown')}")

In [ ]:
# モデルファイル確認
import os
import glob

checkpoints = glob.glob("/content/ComfyUI/models/checkpoints/*")
print("Checkpointsフォルダ内のファイル:")
for f in checkpoints:
    size = os.path.getsize(f)
    print(f"  {os.path.basename(f)}: {size / (1024**3):.2f} GB")

# プロンプト設定
※ ここを変更して生成内容を変更

In [ ]:
# プロンプト設定（ここに生成したい内容を書く）

POSITIVE_PROMPT = """
東京夜景、ネオンライト、赤い傘を持った男性、雨上がり、スーツビジネススーツ、都市の夜景
"""

NEGATIVE_PROMPT = """
blur, low quality, distorted, watermark, cartoon, anime, ugly, deformed
"""

# 生成設定
WIDTH = 2560
HEIGHT = 1440
SEED = 42
STEPS = 8

print(f"Positive: {POSITIVE_PROMPT.strip()}")
print(f"Negative: {NEGATIVE_PROMPT.strip()}")
print(f"Size: {WIDTH}x{HEIGHT}")

In [ ]:
# 画像生成（2560x1440）
import requests
import time

workflow = {
    "1": {
        "inputs": {"ckpt_name": "z-image-turbo-fp8-aio.safetensors"},
        "class_type": "CheckpointLoaderSimple"
    },
    "2": {"inputs": {"text": POSITIVE_PROMPT.strip(), "clip": ["1", 1]}, "class_type": "CLIPTextEncode"},
    "3": {"inputs": {"text": NEGATIVE_PROMPT.strip(), "clip": ["1", 1]}, "class_type": "CLIPTextEncode"},
    "4": {"inputs": {"width": WIDTH, "height": HEIGHT, "batch_size": 1}, "class_type": "EmptyLatentImage"},
    "5": {"inputs": {"model": ["1", 0], "seed": SEED, "steps": STEPS, "cfg": 1.0, "sampler_name": "euler", "scheduler": "normal", "positive": ["2", 0], "negative": ["3", 0], "latent_image": ["4", 0], "denoise": 1.0}, "class_type": "KSampler"},
    "6": {"inputs": {"samples": ["5", 0], "vae": ["1", 2]}, "class_type": "VAEDecode"},
    "7": {"inputs": {"images": ["6", 0], "filename_prefix": "result"}, "class_type": "SaveImage"}
}

print(f"Generating {WIDTH}x{HEIGHT} image... (最大20分)")
result = requests.post("http://127.0.0.1:8188/prompt", json={"prompt": workflow}).json()

if "prompt_id" in result:
    prompt_id = result['prompt_id']
    print(f"Prompt ID: {prompt_id}")
    
    for i in range(1200):
        time.sleep(1)
        hist = requests.get(f"http://127.0.0.1:8188/history/{prompt_id}").json()
        if prompt_id in hist and hist[prompt_id]['status']['completed']:
            print("Complete!")
            break
        if i % 60 == 0:
            print(f"Progress: {i}s ({i//60}分)...")
    else:
        print("Timeout! 画像生成に時間がかかりすぎています。")
else:
    print(f"Error: {result}")

In [ ]:
# PNG → JPG変換（高品質）
from PIL import Image
import glob
import os

# 最新PNGを検索
png_files = glob.glob("/content/ComfyUI/output/result_*.png")
if png_files:
    latest_png = max(png_files, key=os.path.getmtime)
    jpg_path = latest_png.replace(".png", ".jpg")
    
    # 高品質JPGに変換（quality 95はShutterstock推奨）
    img = Image.open(latest_png)
    if img.mode in ('RGBA', 'P'):
        img = img.convert('RGB')
    img.save(jpg_path, 'JPEG', quality=95, optimize=True)
    
    print(f"変換完了: {jpg_path}")
    print(f"JPGファイルサイズ: {os.path.getsize(jpg_path) / (1024*1024):.1f} MB")
else:
    print("PNGファイルが見つかりません")

In [ ]:
# JPG画像ダウンロード
from google.colab import files
import glob
import os

jpg_files = glob.glob("/content/ComfyUI/output/result_*.jpg")
if jpg_files:
    latest = max(jpg_files, key=os.path.getmtime)
    print(f"Downloading: {latest}")
    files.download(latest)
else:
    print("No JPG found")